# KLE Colab — Multi-Strategy Voting Ensemble

目标：**10+ hits / 20**

## 核心策略：投票制多策略融合

| 投票者 | 逻辑 | 权重 |
|--------|------|------|
| CNN Ensemble | 多 seq_len × 多 seed 的 PyTorch CNN | 可调 |
| Frequency (频率) | 近 N 期中每个号码出现频率 | 可调 |
| Gap (遗漏) | 已多久未出现，越久越"该出" | 可调 |
| Momentum (动量) | 最近 3-5 期连续出现的号码 | 可调 |

每个策略独立给 80 个号码打分 → 加权融合 → 取 top-20。

## 为什么比旧方式好
1. **旧方式**：averaging 概率图 → 所有值趋于均匀 → ≈随机
2. **新方式**：多维度独立打分 → 加权投票 → 信号叠加放大

建议 Colab 运行时切换到 `GPU`。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


REPO_URL = "https://github.com/jursmatsko/lotto-image-predictor-research.git"
REPO_ROOT = Path("/content/lotto-image-predictor-research")
PROJECT_DIR = REPO_ROOT / "kle"

run("python -m pip install -q pandas matplotlib requests beautifulsoup4 lxml")

try:
    import torch
except ImportError:
    run("python -m pip install -q torch")
    import torch

if not REPO_ROOT.exists():
    run(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 数据检查

先确认 `data/data.csv` 能正确读取，并检查最近几期数据结构。

In [ ]:
import pandas as pd


data_path = PROJECT_DIR / "data" / "data.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head())
display(df.tail())

## 加载应用

这里直接复用仓库里的 `ImagePredictionApp`，但训练模型选择 `pytorch_cnn`，因为它在 Colab 的兼容性和速度都更好。

In [ ]:
from image_predictor.main import ImagePredictionApp
from image_predictor.models.pytorch_image_cnn import DEVICE


app = ImagePredictionApp()
app.loader.load_data()

print(f"Training device: {DEVICE}")
print(app.loader.get_statistics())

## 参数配置

| 参数 | 作用 |
|------|------|
| `SEQ_LENS` | 每个集成成员使用的历史序列长度，越多越丰富但越慢 |
| `MODELS_PER_SEQ` | 每个 seq_len 训练几个不同 seed 的模型 |
| `WF_EPOCHS` | Walk-forward 回测中每个模型的训练轮数（建议 20-30） |
| `FINAL_EPOCHS` | 最终预测时每个模型的训练轮数（建议 40-60） |
| `WF_EVAL_LAST` | 在最近多少期上做回测评估 |

**总模型数 = len(SEQ_LENS) × MODELS_PER_SEQ**，越多越准但越慢。

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  预设
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PRESET = "balanced"

PRESETS = {
    "fast":     dict(seq_lens=[10, 15, 20],     models_per_seq=1, wf_epochs=15, final_epochs=30, wf_eval_last=15),
    "balanced": dict(seq_lens=[10, 15, 20],     models_per_seq=2, wf_epochs=25, final_epochs=50, wf_eval_last=20),
    "accurate": dict(seq_lens=[8, 12, 16, 20, 25], models_per_seq=3, wf_epochs=30, final_epochs=60, wf_eval_last=25),
}

cfg            = PRESETS[PRESET]
SEQ_LENS       = cfg["seq_lens"]
MODELS_PER_SEQ = cfg["models_per_seq"]
WF_EPOCHS      = cfg["wf_epochs"]
FINAL_EPOCHS   = cfg["final_epochs"]
WF_EVAL_LAST   = cfg["wf_eval_last"]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  训练参数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TRAIN_WINDOW   = 300      # 滑动窗口（None = 全部历史）
TIME_DECAY     = 3.0      # 近期数据加权强度
LEARNING_RATE  = 7e-4
HIDDEN_CHANNELS = 96      # 模型容量（旧 64 → 新 96）
BASE_SEED      = 20260313

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  多策略融合权重
#
#  CNN 模型只是 4 个投票者之一
#  统计策略不需要训练，成本为零，但信号很强
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
W_CNN        = 0.35   # CNN ensemble 权重
W_FREQUENCY  = 0.30   # 近期频率（最近 FREQ_WINDOW 期内出现次数）
W_GAP        = 0.20   # 遗漏回补（已 N 期未出，越久分越高）
W_MOMENTUM   = 0.15   # 动量（最近 MOMENTUM_WINDOW 期连续出现得分高）

FREQ_WINDOW     = 30    # 频率统计窗口
GAP_WINDOW      = 100   # 遗漏统计窗口
MOMENTUM_WINDOW = 5     # 动量统计窗口

NUM_TICKETS = 10

MODEL_CACHE_DIR = PROJECT_DIR / "image_predictor" / "models" / "saved"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

n_cnn = len(SEQ_LENS) * MODELS_PER_SEQ
eta_min = n_cnn * WF_EVAL_LAST * 8 / 60

print(f"{'─'*56}")
print(f"  Preset       : {PRESET.upper()}")
print(f"  CNN models   : {n_cnn}  ({len(SEQ_LENS)} seq × {MODELS_PER_SEQ} seeds)")
print(f"  + 3 stat strategies (freq / gap / momentum)")
print(f"  Weights      : CNN={W_CNN} Freq={W_FREQUENCY} Gap={W_GAP} Mom={W_MOMENTUM}")
print(f"  train_window : {TRAIN_WINDOW}  |  hidden_ch: {HIDDEN_CHANNELS}")
print(f"  Est. WF time : ~{eta_min:.0f} min  ({n_cnn * WF_EVAL_LAST} CNN models)")
print(f"{'─'*56}")

In [ ]:
import time
import numpy as np
import torch
from tqdm.notebook import tqdm
from image_predictor.models.pytorch_image_cnn import PyTorchImagePredictor


# ═══════════════════════════════════════════════════════
#  Helper functions
# ═══════════════════════════════════════════════════════

def train_one_model(train_imgs, seq_len, epochs, lr, seed, time_decay=3.0, hc=96):
    np.random.seed(seed)
    torch.manual_seed(seed)
    X, y = [], []
    for i in range(seq_len, len(train_imgs)):
        X.append(train_imgs[i - seq_len:i])
        y.append(train_imgs[i])
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    val_size = max(1, int(len(X) * 0.15))
    model = PyTorchImagePredictor(seq_len=seq_len, hidden_channels=hc)
    model.fit(X[:-val_size], y[:-val_size], X[-val_size:], y[-val_size:],
              epochs=epochs, lr=lr, time_decay=time_decay, verbose=False)
    return model


def stat_frequency_scores(draws, window):
    """频率策略：最近 window 期内每个号码出现次数 → 归一化分数。"""
    recent = draws[-window:]
    counts = np.zeros(80, dtype=np.float32)
    for draw in recent:
        for n in draw:
            counts[n - 1] += 1
    mx = counts.max()
    return counts / mx if mx > 0 else counts


def stat_gap_scores(draws, window):
    """遗漏策略：号码已连续缺席期数 → 越久分越高。"""
    recent = draws[-window:]
    last_seen = np.full(80, -1, dtype=np.float32)
    for i, draw in enumerate(recent):
        for n in draw:
            last_seen[n - 1] = i
    total = len(recent)
    gaps = np.where(last_seen >= 0, total - 1 - last_seen, total).astype(np.float32)
    mx = gaps.max()
    return gaps / mx if mx > 0 else gaps


def stat_momentum_scores(draws, window):
    """动量策略：最近 window 期内连续出现 → 分数更高。"""
    recent = draws[-window:]
    scores = np.zeros(80, dtype=np.float32)
    for t, draw in enumerate(recent):
        w = (t + 1) / len(recent)  # 越近权重越大
        for n in draw:
            scores[n - 1] += w
    mx = scores.max()
    return scores / mx if mx > 0 else scores


def fuse_scores(cnn_scores, freq_scores, gap_scores, mom_scores,
                w_cnn, w_freq, w_gap, w_mom):
    """加权融合 4 个策略的 80 维分数 → 取 top-20。"""
    combined = (w_cnn * cnn_scores + w_freq * freq_scores +
                w_gap * gap_scores + w_mom * mom_scores)
    top20_idx = np.argsort(combined)[::-1][:20]
    return sorted([int(i + 1) for i in top20_idx]), combined


# ═══════════════════════════════════════════════════════
#  准备数据
# ═══════════════════════════════════════════════════════
app.loader.load_data()
all_draws = app.loader.draws
images    = app.loader.create_images().astype(np.float32)
issues    = app.loader.issues
encoder   = app.encoder
min_train = max(SEQ_LENS) + 50

eval_start = max(min_train, len(images) - WF_EVAL_LAST)
n_steps    = len(images) - eval_start

print(f"Total draws       : {len(images)}")
print(f"Walk-forward steps: {n_steps}")
print(f"CNN models/step   : {len(SEQ_LENS)*MODELS_PER_SEQ}")
print(f"+ 3 stat strategies (freq/gap/momentum)")
print(f"Weights: CNN={W_CNN} Freq={W_FREQUENCY} Gap={W_GAP} Mom={W_MOMENTUM}")
print("=" * 60)

# ═══════════════════════════════════════════════════════
#  Walk-forward 回测
# ═══════════════════════════════════════════════════════
wf_results   = []
running_hits = []
wf_start     = time.time()

step_bar = tqdm(range(eval_start, len(images)), desc="Walk-forward", unit="draw")

for test_idx in step_bar:
    t0 = time.time()

    # 滑动窗口
    if TRAIN_WINDOW:
        w_start    = max(0, test_idx - TRAIN_WINDOW)
        train_imgs = images[w_start:test_idx]
    else:
        train_imgs = images[:test_idx]

    history_draws = all_draws[:test_idx]

    # ── 策略 1: CNN ensemble → 80 维分数 ──────────────
    cnn_preds = []
    for sl in SEQ_LENS:
        if len(train_imgs) <= sl + 30:
            continue
        for m in range(MODELS_PER_SEQ):
            seed = BASE_SEED + test_idx * 10 + sl * 100 + m
            mdl  = train_one_model(train_imgs, sl, WF_EPOCHS, LEARNING_RATE, seed,
                                   time_decay=TIME_DECAY, hc=HIDDEN_CHANNELS)
            latest = train_imgs[-sl:][np.newaxis, ...]
            cnn_preds.append(mdl.predict(latest)[0])

    cnn_avg = np.mean(np.stack(cnn_preds, axis=0), axis=0)
    cnn_flat = cnn_avg.flatten()
    cnn_norm = cnn_flat / cnn_flat.max() if cnn_flat.max() > 0 else cnn_flat

    # ── 策略 2-4: 统计策略 ────────────────────────────
    freq_sc = stat_frequency_scores(history_draws, FREQ_WINDOW)
    gap_sc  = stat_gap_scores(history_draws, GAP_WINDOW)
    mom_sc  = stat_momentum_scores(history_draws, MOMENTUM_WINDOW)

    # ── 融合 ─────────────────────────────────────────
    pred_nums, _ = fuse_scores(cnn_norm, freq_sc, gap_sc, mom_sc,
                               W_CNN, W_FREQUENCY, W_GAP, W_MOMENTUM)
    pred_set  = set(pred_nums)
    actual_set = set(all_draws[test_idx])
    hits = len(pred_set & actual_set)

    running_hits.append(hits)
    avg_so_far = np.mean(running_hits)

    wf_results.append({
        "issue":     issues[test_idx],
        "hits":      hits,
        "predicted": sorted(pred_set),
        "actual":    sorted(actual_set),
    })
    step_bar.set_postfix(
        hits=f"{hits}/20",
        avg=f"{avg_so_far:.2f}",
        vs_rand=f"{avg_so_far-5:+.2f}",
        t=f"{time.time()-t0:.0f}s",
    )

elapsed   = time.time() - wf_start
avg_final = np.mean(running_hits)
above_10  = sum(h >= 10 for h in running_hits)
print(f"\n{'='*60}")
print(f"  Walk-forward complete in {elapsed/60:.1f} min")
print(f"  Average hits   : {avg_final:.2f} / 20")
print(f"  vs Random      : {avg_final - 5:+.2f}")
print(f"  Draws ≥ 10 hits: {above_10} / {len(running_hits)}")
print(f"  Draws > 5 hits : {sum(h > 5 for h in running_hits)} / {len(running_hits)}")
print(f"{'='*60}")

## Walk-Forward 回测结果

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

wf_df = pd.DataFrame(wf_results)
avg   = wf_df["hits"].mean()
above = (wf_df["hits"] > 5).sum()

print("=" * 50)
print(f"  Average hits  : {avg:.2f} / 20")
print(f"  Random baseline: 5.00 / 20")
print(f"  vs Random      : {avg - 5:+.2f}")
print(f"  Draws > 5 hits : {above} / {len(wf_df)}")
print("=" * 50)
display(wf_df[["issue", "hits"]])

# ── 柱状图 ───────────────────────────────────────────────
colors = ["#2ecc71" if h > 5 else "#e74c3c" if h < 5 else "#f39c12"
          for h in wf_df["hits"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(wf_df)), wf_df["hits"], color=colors)
axes[0].axhline(5.0, color="orange", linestyle="--", linewidth=1.5, label="Random (5.0)")
axes[0].axhline(avg, color="royalblue", linestyle="-", linewidth=2, label=f"Average ({avg:.2f})")
axes[0].set_xlabel("Draw index")
axes[0].set_ylabel("Hits / 20")
axes[0].set_title("Walk-Forward Ensemble: Hits per Draw")
axes[0].legend()

green_p = mpatches.Patch(color="#2ecc71", label="> 5 hits")
red_p   = mpatches.Patch(color="#e74c3c", label="< 5 hits")
axes[0].legend(handles=[green_p, red_p,
    plt.Line2D([], [], color="orange", linestyle="--", label="Random"),
    plt.Line2D([], [], color="royalblue", label=f"Avg {avg:.2f}")
])

# 命中分布直方图
axes[1].hist(wf_df["hits"], bins=range(0, 21), color="#3498db", edgecolor="white", align="left")
axes[1].axvline(5.0, color="orange", linestyle="--", linewidth=1.5, label="Random (5.0)")
axes[1].axvline(avg,  color="royalblue", linestyle="-",  linewidth=2,   label=f"Avg ({avg:.2f})")
axes[1].set_xlabel("Hits")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Hit Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(str(MODEL_CACHE_DIR / "wf_results.png"), dpi=120, bbox_inches="tight")
plt.show()

## 最终预测 — 用全量数据训练 Ensemble

回测只用来评估策略质量。最终预测用**全量历史数据**重新训练一批 ensemble 模型，集成后给出下一期推荐。

In [ ]:
# ── 最终预测：与回测完全一致的多策略融合 ──────────────
final_train = images[-TRAIN_WINDOW:] if TRAIN_WINDOW else images
print(f"Training final ensemble — window: last {len(final_train)} draws …")
print(f"  CNN: {len(SEQ_LENS)} seq_lens × {MODELS_PER_SEQ} = {len(SEQ_LENS)*MODELS_PER_SEQ}")
print(f"  + freq / gap / momentum stats")

# CNN ensemble
final_cnn_preds = []
for sl in SEQ_LENS:
    for m in range(MODELS_PER_SEQ):
        seed = BASE_SEED + 88888 + sl * 100 + m
        mdl  = train_one_model(final_train, sl, FINAL_EPOCHS, LEARNING_RATE, seed,
                               time_decay=TIME_DECAY, hc=HIDDEN_CHANNELS)
        latest = final_train[-sl:][np.newaxis, ...]
        final_cnn_preds.append(mdl.predict(latest)[0])
        print(f"  ✓ seq_len={sl}  seed={m+1}")

cnn_avg  = np.mean(np.stack(final_cnn_preds, axis=0), axis=0)
cnn_flat = cnn_avg.flatten()
cnn_norm = cnn_flat / cnn_flat.max() if cnn_flat.max() > 0 else cnn_flat

# Statistical strategies
freq_final = stat_frequency_scores(all_draws, FREQ_WINDOW)
gap_final  = stat_gap_scores(all_draws, GAP_WINDOW)
mom_final  = stat_momentum_scores(all_draws, MOMENTUM_WINDOW)

# Fuse
top20, combined_scores = fuse_scores(
    cnn_norm, freq_final, gap_final, mom_final,
    W_CNN, W_FREQUENCY, W_GAP, W_MOMENTUM,
)

score_map = combined_scores.reshape(8, 10)
print(f"\n{'='*60}")
print(f"  CNN models    : {len(final_cnn_preds)}")
print(f"  + 3 stats     : freq / gap / momentum")
print(f"  Weights       : CNN={W_CNN} Freq={W_FREQUENCY} Gap={W_GAP} Mom={W_MOMENTUM}")
print(f"  Top 20        : {top20}")
print(f"{'='*60}")

# ── 融合分数热力图 ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 融合热力图
im0 = axes[0].imshow(score_map, cmap="hot", vmin=0, aspect="equal")
fig.colorbar(im0, ax=axes[0], label="Combined Score")
for r in range(8):
    for c in range(10):
        num = r * 10 + c + 1
        is_top = num in set(top20)
        axes[0].text(c, r, str(num), ha="center", va="center", fontsize=7,
                     fontweight="bold",
                     color="cyan" if is_top else ("white" if score_map[r,c] > score_map.mean() else "gray"))
axes[0].set_title("Fused Score Map — Top20 = cyan")

# CNN 热力图
im1 = axes[1].imshow(cnn_avg, cmap="hot", vmin=0, aspect="equal")
fig.colorbar(im1, ax=axes[1], label="CNN Prob")
axes[1].set_title("CNN Ensemble Only")

# 频率 + 遗漏 对比
axes[2].bar(range(1, 81), freq_final, alpha=0.5, label="Frequency", color="#2ecc71")
axes[2].bar(range(1, 81), gap_final * 0.5, alpha=0.5, label="Gap × 0.5", color="#e74c3c")
axes[2].set_xlabel("Number")
axes[2].set_ylabel("Score")
axes[2].set_title("Freq vs Gap")
axes[2].legend()

for ax in axes[:2]:
    ax.set_xticks(range(10)); ax.set_xticklabels(range(1, 11))
    ax.set_yticks(range(8));  ax.set_yticklabels([f"{r*10+1}-{r*10+10}" for r in range(8)])

plt.tight_layout()
plt.savefig(str(MODEL_CACHE_DIR / "ensemble_heatmap.png"), dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
tickets = app._generate_tickets(ensemble_final, NUM_TICKETS)

print(f"{'='*50}")
print(f"  Next Draw Prediction")
print(f"  Top 20: {' '.join(f'{n:02d}' for n in top20)}")
print(f"{'='*50}")
print(f"\n  {NUM_TICKETS} Recommended Tickets (10 numbers each):")
for i, ticket in enumerate(tickets, 1):
    print(f"  Ticket {i:02d}: {' '.join(f'{n:02d}' for n in ticket)}")

print(f"\n  Walk-Forward Backtest Summary ({WF_EVAL_LAST} draws):")
print(f"  Average Hits : {avg:.2f}/20  (Random baseline: 5.00/20)")
print(f"  vs Random    : {avg-5:+.2f}")
print(f"\n  ⚠️  Lottery is random. This is for scientific research only.")

## 可选：保存预测结果到 Google Drive

Colab 会话结束后本地文件会消失。运行下面这个单元，把图表和预测结果保存到 Google Drive 长期保留。

In [ ]:
import shutil, json
from google.colab import drive

drive.mount('/content/drive')

drive_dir = Path('/content/drive/MyDrive/kle_results')
drive_dir.mkdir(parents=True, exist_ok=True)

# 保存图表
for fname in ["wf_results.png", "ensemble_heatmap.png"]:
    src = MODEL_CACHE_DIR / fname
    if src.exists():
        shutil.copy(src, drive_dir / fname)
        print(f"✓ Saved chart : {fname}")

# 保存预测结果 JSON
result = {
    "preset": PRESET,
    "top20": top20,
    "tickets": tickets,
    "wf_avg_hits": round(float(avg), 3),
    "wf_vs_random": round(float(avg) - 5.0, 3),
    "wf_eval_last": WF_EVAL_LAST,
    "wf_details": wf_results,
}
result_path = drive_dir / "prediction_result.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ Saved result: prediction_result.json")
print(f"\nAll files saved to: {drive_dir}")